# Exposure -> Conversion Join and Attribution Window Comparison

## Beginner-friendly summary
This notebook starts from raw marketing events and builds a measurement-ready attribution view. It focuses on getting the exposure->conversion join right first, then testing how attribution choices change channel credit.

### What this notebook does
- Uses a real-world dataset: Criteo Attribution Modeling Dataset
- Builds explicit exposure->conversion joins with auditable rules
- Compares attribution methods: last-touch, first-touch, linear, and time-decay
- Tests sensitivity by attribution windows, lag filters, and user/channel segments
- Produces attributed vs unattributed conversion diagnostics, matched vs unmatched diagnostics, and channel-level attributed revenue

### Major steps (in order)
1. Load real event data
2. Build exposure and conversion tables
3. Join events by window and lag rules
4. Compare attribution methods
5. Measure attributed vs unattributed outcomes and matching quality
6. Run robustness and segment sensitivity checks
7. Export a measurement-ready output table

### Side notes for beginners
- A join window answers: "how far back can an exposure get credit?"
- Attribution redistributes observed credit; it does not prove causal incrementality.
- If results change sharply across windows/rules, decision risk is higher.
- Keep this notebook for measurement logic; keep `marketing_attribution.ipynb` for high-level ROI/ROAS/CPA reporting.


## 1) Setup

> Side note: We keep dependencies lightweight (`pandas`, `numpy`) so the notebook is easy to run.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

## 2) Download Real-World Dataset (Criteo Attribution)

Dataset source:
- [Criteo Attribution Modeling Dataset](https://huggingface.co/datasets/criteo/criteo-attribution-dataset)

Direct file used in this notebook:
- `criteo_attribution_dataset.tsv.gz`

> Side note: The raw file is large. We read a configurable row sample for fast iteration.

In [2]:
import urllib.request

data_dir = Path('data')
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / 'criteo_attribution_dataset.tsv.gz'
dataset_url = 'https://huggingface.co/datasets/criteo/criteo-attribution-dataset/resolve/main/criteo_attribution_dataset.tsv.gz'

if not dataset_path.exists():
    print('Downloading real-world dataset (this can take time)...')
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print('Download complete:', dataset_path)
else:
    print('Dataset already present:', dataset_path)

Dataset already present: data/criteo_attribution_dataset.tsv.gz


## 3) Load and Prepare Event Tables

Expected raw columns (tab-separated, no header):
- `timestamp`
- `uid`
- `campaign`
- `conversion`
- `conversion_timestamp`

> Side note: A single user may have many touchpoints and multiple conversions.

In [3]:
TARGET_CONVERTED_USERS = 2000
CHUNK_SIZE = 500_000
cols = ['timestamp', 'uid', 'campaign', 'conversion', 'conversion_timestamp', 'conversion_id']

# Pass 1: collect users who have at least one conversion event
converted_users = set()
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['conversion'] = pd.to_numeric(chunk['conversion'], errors='coerce').fillna(0).astype(int)
    conv = chunk[chunk['conversion'] == 1]
    if not conv.empty:
        converted_users.update(conv['uid'].dropna().astype(str).tolist())
    if len(converted_users) >= TARGET_CONVERTED_USERS:
        break

selected_users = set(list(converted_users)[:TARGET_CONVERTED_USERS])
print('Selected converted users:', len(selected_users))

# Pass 2: collect all touchpoints for selected users
parts = []
for chunk in pd.read_csv(dataset_path, sep='\t', compression='gzip', usecols=cols, chunksize=CHUNK_SIZE, low_memory=False):
    chunk['uid'] = chunk['uid'].astype(str)
    keep = chunk[chunk['uid'].isin(selected_users)]
    if not keep.empty:
        parts.append(keep)

if not parts:
    raise RuntimeError('No rows found for selected converted users. Try increasing TARGET_CONVERTED_USERS.')

raw = pd.concat(parts, ignore_index=True)

raw['timestamp'] = pd.to_numeric(raw['timestamp'], errors='coerce')
raw['conversion_timestamp'] = pd.to_numeric(raw['conversion_timestamp'], errors='coerce')
raw['conversion'] = pd.to_numeric(raw['conversion'], errors='coerce').fillna(0).astype(int)
raw['conversion_id'] = pd.to_numeric(raw['conversion_id'], errors='coerce')
raw = raw.dropna(subset=['timestamp', 'uid', 'campaign']).copy()
raw['ts_exposure'] = pd.to_datetime(raw['timestamp'], unit='s', errors='coerce')

# Exposure table
exposures = raw[['uid', 'campaign', 'timestamp', 'ts_exposure']].rename(columns={'uid': 'user_id', 'campaign': 'channel'})
exposures['exposure_id'] = np.arange(len(exposures), dtype=np.int64)

# Conversion table: one row per distinct conversion event
conv_rows = raw[(raw['conversion'] == 1) & (raw['conversion_timestamp'].notna())].copy()
conv_rows = conv_rows[conv_rows['conversion_timestamp'] >= conv_rows['timestamp']]

conversions = conv_rows[['uid', 'conversion_id', 'conversion_timestamp']].rename(columns={'uid': 'user_id'}).drop_duplicates().reset_index(drop=True)
conversions['ts_conversion'] = pd.to_datetime(conversions['conversion_timestamp'], unit='s', errors='coerce')
conversions = conversions.dropna(subset=['ts_conversion']).reset_index(drop=True)
conversions['conversion_id'] = np.arange(len(conversions), dtype=np.int64)
conversions['conversion_value'] = 1.0

print('Filtered raw rows:', len(raw))
print('Exposure rows:', len(exposures))
print('Conversion rows:', len(conversions))
print('Unique users in filtered set:', exposures['user_id'].nunique())

display(exposures.head(5))
display(conversions.head(5))


Selected converted users: 2000


Filtered raw rows: 26500
Exposure rows: 26500
Conversion rows: 3800
Unique users in filtered set: 2000


,user_id,channel,timestamp,ts_exposure,exposure_id
0,7306395,29427842,3,1970-01-01 00:00:03,0
1,6013772,15184511,25,1970-01-01 00:00:25,1
2,16502720,23817046,244,1970-01-01 00:04:04,2
3,18997477,32398758,397,1970-01-01 00:06:37,3
4,12324598,32368244,705,1970-01-01 00:11:45,4


,user_id,conversion_id,conversion_timestamp,ts_conversion,conversion_value
0,7306395,0,1449193,1970-01-17 18:33:13,1.0
1,6013772,1,138469,1970-01-02 14:27:49,1.0
2,16502720,2,257000,1970-01-03 23:23:20,1.0
3,18997477,3,255090,1970-01-03 22:51:30,1.0
4,12324598,4,116423,1970-01-02 08:20:23,1.0


## 4) Exposure -> Conversion Join (Windowed)

Join rule:
- same `user_id`
- exposure occurred before conversion
- exposure within lookback window (in days)

> Side note: This is the auditable core of measurement pipelines.

In [4]:
def build_join_table(exposures_df: pd.DataFrame, conversions_df: pd.DataFrame, lookback_days: int) -> pd.DataFrame:
    merged = conversions_df.merge(exposures_df, on='user_id', how='left')
    delta_hours = (merged['ts_conversion'] - merged['ts_exposure']).dt.total_seconds() / 3600

    valid = merged[(delta_hours >= 0) & (delta_hours <= lookback_days * 24)].copy()
    if valid.empty:
        return pd.DataFrame(columns=[
            'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
            'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
        ])

    valid['hours_before_conversion'] = delta_hours.loc[valid.index]
    valid['lookback_days'] = lookback_days

    cols = [
        'lookback_days', 'conversion_id', 'user_id', 'ts_conversion', 'conversion_value',
        'exposure_id', 'channel', 'ts_exposure', 'hours_before_conversion'
    ]
    return valid[cols].sort_values(['conversion_id', 'ts_exposure']).reset_index(drop=True)

joined_7d = build_join_table(exposures, conversions, lookback_days=7)
print('Joined rows (7-day):', len(joined_7d))
print('Unique conversions covered (7-day):', joined_7d['conversion_id'].nunique())
print('Coverage % (7-day):', round(100 * joined_7d['conversion_id'].nunique() / max(len(conversions), 1), 2))
display(joined_7d.head(10))


Joined rows (7-day): 17187
Unique conversions covered (7-day): 3395
Coverage % (7-day): 89.34


,lookback_days,conversion_id,user_id,ts_conversion,conversion_value,exposure_id,channel,ts_exposure,hours_before_conversion
0,7,0,7306395,1970-01-17 18:33:13,1.0,12299,29427842,1970-01-10 22:32:07,164.018333
1,7,0,7306395,1970-01-17 18:33:13,1.0,13147,29427842,1970-01-12 10:14:25,128.313333
2,7,0,7306395,1970-01-17 18:33:13,1.0,14882,29427842,1970-01-14 10:14:41,80.308889
3,7,0,7306395,1970-01-17 18:33:13,1.0,17027,29427842,1970-01-16 23:31:42,19.025278
4,7,0,7306395,1970-01-17 18:33:13,1.0,17514,29427842,1970-01-17 16:21:13,2.200000
5,7,1,6013772,1970-01-02 14:27:49,1.0,1,15184511,1970-01-01 00:00:25,38.456667
6,7,1,6013772,1970-01-02 14:27:49,1.0,1890,15184511,1970-01-01 14:15:22,24.207500
7,7,1,6013772,1970-01-02 14:27:49,1.0,3697,15184511,1970-01-02 09:09:54,5.298611
8,7,1,6013772,1970-01-02 14:27:49,1.0,3702,15184511,1970-01-02 09:14:57,5.214444
9,7,1,6013772,1970-01-02 14:27:49,1.0,4158,15184511,1970-01-02 14:27:30,0.005278


## 4b) Coverage Summary (Explicit)

> Side note: Coverage tells us what fraction of conversions can be linked to at least one prior exposure in the selected lookback window.


In [5]:
total_conversions = int(conversions['conversion_id'].nunique())
matched_conversions_7d = int(joined_7d['conversion_id'].nunique())
coverage_pct_7d = round(100 * matched_conversions_7d / max(total_conversions, 1), 2)

coverage_summary = pd.DataFrame([{
    'lookback_days': 7,
    'total_conversions': total_conversions,
    'matched_conversions': matched_conversions_7d,
    'coverage_pct': coverage_pct_7d
}])

print(f"Coverage formula: matched_conversions / total_conversions = {matched_conversions_7d} / {total_conversions} = {coverage_pct_7d}%")
display(coverage_summary)


Coverage formula: matched_conversions / total_conversions = 3395 / 3800 = 89.34%


,lookback_days,total_conversions,matched_conversions,coverage_pct
0,7,3800,3395,89.34


## 4c) Attributed vs Unattributed and Matched vs Unmatched (Explicit)

> Side note: This makes coverage gaps operationally visible before attribution comparisons.


In [6]:
# Conversion-level attribution coverage
total_conversions = int(conversions['conversion_id'].nunique())
attributed_conversions = int(joined_7d['conversion_id'].nunique())
unattributed_conversions = int(max(total_conversions - attributed_conversions, 0))

# User/event matching coverage
all_users = set(exposures['user_id'].astype(str).unique()) | set(conversions['user_id'].astype(str).unique())
matched_users = set(joined_7d['user_id'].astype(str).unique())
unmatched_users = all_users - matched_users

total_exposure_events = int(len(exposures))
matched_exposure_events = int(joined_7d['exposure_id'].nunique())
unmatched_exposure_events = int(max(total_exposure_events - matched_exposure_events, 0))

coverage_ops = pd.DataFrame([
    {'metric': 'conversions_total', 'value': total_conversions},
    {'metric': 'conversions_attributed_7d', 'value': attributed_conversions},
    {'metric': 'conversions_unattributed_7d', 'value': unattributed_conversions},
    {'metric': 'users_total_union', 'value': len(all_users)},
    {'metric': 'users_matched_7d', 'value': len(matched_users)},
    {'metric': 'users_unmatched_7d', 'value': len(unmatched_users)},
    {'metric': 'exposure_events_total', 'value': total_exposure_events},
    {'metric': 'exposure_events_matched_7d', 'value': matched_exposure_events},
    {'metric': 'exposure_events_unmatched_7d', 'value': unmatched_exposure_events},
])

display(coverage_ops)


,metric,value
0,conversions_total,3800
1,conversions_attributed_7d,3395
2,conversions_unattributed_7d,405
3,users_total_union,2000
4,users_matched_7d,1821
5,users_unmatched_7d,179
6,exposure_events_total,26500
7,exposure_events_matched_7d,11281
8,exposure_events_unmatched_7d,15219


## 5) Attribution Method Comparison (Expanded)

Methods included:
- Last-touch attribution
- First-touch attribution
- Linear multi-touch attribution
- Time-decay attribution
- U-shape (position-based)
- W-shape (position-based)
- Markov-style removal proxy
- Shapley-style proxy

> Side note: Markov/Shapley entries here are practical proxy implementations for measurement workflow learning.


In [7]:
def _normalize_weights(s: pd.Series) -> pd.Series:
    total = float(s.sum())
    if total <= 0:
        n = len(s)
        return pd.Series(np.ones(n) / max(n, 1), index=s.index)
    return s / total

def _position_rank(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(['conversion_id', 'ts_exposure']).copy()
    out['pos'] = out.groupby('conversion_id').cumcount()
    out['n'] = out.groupby('conversion_id')['exposure_id'].transform('count')
    return out

def attrib_last_touch(join_df: pd.DataFrame) -> pd.DataFrame:
    idx = join_df.groupby('conversion_id')['ts_exposure'].idxmax()
    out = join_df.loc[idx, ['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_first_touch(join_df: pd.DataFrame) -> pd.DataFrame:
    idx = join_df.groupby('conversion_id')['ts_exposure'].idxmin()
    out = join_df.loc[idx, ['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_linear(join_df: pd.DataFrame) -> pd.DataFrame:
    cnt = join_df.groupby('conversion_id')['exposure_id'].transform('count')
    out = join_df[['channel', 'conversion_value']].copy()
    out['credit'] = out['conversion_value'] / cnt
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_time_decay(join_df: pd.DataFrame, half_life_hours: float = 48.0) -> pd.DataFrame:
    out = join_df[['conversion_id', 'channel', 'conversion_value', 'hours_before_conversion']].copy()
    out['w'] = np.exp(-np.log(2) * out['hours_before_conversion'] / half_life_hours)
    out['w'] = out.groupby('conversion_id')['w'].transform(lambda x: _normalize_weights(x))
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_u_shape(join_df: pd.DataFrame) -> pd.DataFrame:
    out = _position_rank(join_df)
    pos = out['pos']
    n = out['n']
    w = np.where(
        n == 1,
        1.0,
        np.where(
            n == 2,
            0.5,
            np.where(pos == 0, 0.4, np.where(pos == (n - 1), 0.4, 0.2 / (n - 2)))
        )
    )
    out['w'] = w
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_w_shape(join_df: pd.DataFrame) -> pd.DataFrame:
    out = _position_rank(join_df)
    pos = out['pos']
    n = out['n']
    mid = (n // 2)
    anchor = (pos == 0) | (pos == (n - 1)) | (pos == mid)
    anchor_count = anchor.groupby(out['conversion_id']).transform('sum').astype(float)
    rem_count = (n - anchor_count).astype(float)

    w_main = np.where(anchor, 0.9 / np.maximum(anchor_count, 1.0), np.where(rem_count > 0, 0.1 / rem_count, 0.0))
    w_small = 1.0 / np.maximum(n, 1)
    w = np.where(n <= 2, w_small, w_main)

    out['w'] = w
    out['credit'] = out['conversion_value'] * out['w']
    return out.groupby('channel', as_index=False)['credit'].sum()

def attrib_markov_proxy(join_df: pd.DataFrame) -> pd.DataFrame:
    # Proxy: channel removal effect approximated by conversion-path participation share.
    paths = join_df.groupby('conversion_id')['channel'].apply(lambda x: tuple(pd.unique(x)))
    channels = sorted(join_df['channel'].dropna().unique())
    scores = []
    for ch in channels:
        affected = paths.apply(lambda p: ch in p).sum()
        scores.append((ch, float(affected)))
    out = pd.DataFrame(scores, columns=['channel', 'score'])
    out['score'] = _normalize_weights(out['score'])
    total_val = float(join_df.drop_duplicates('conversion_id')['conversion_value'].sum())
    out['credit'] = out['score'] * total_val
    return out[['channel', 'credit']]

def attrib_shapley_proxy(join_df: pd.DataFrame) -> pd.DataFrame:
    # Proxy: per-conversion unique-channel sharing (Shapley-style equal marginal under symmetric coalition assumption).
    temp = join_df.groupby('conversion_id').agg(
        conversion_value=('conversion_value', 'first'),
        channels=('channel', lambda x: list(pd.unique(x)))
    ).reset_index()
    rows = []
    for _, r in temp.iterrows():
        chs = [c for c in r['channels'] if pd.notna(c)]
        if not chs:
            continue
        share = float(r['conversion_value']) / len(chs)
        for ch in chs:
            rows.append((ch, share))
    out = pd.DataFrame(rows, columns=['channel', 'credit'])
    return out.groupby('channel', as_index=False)['credit'].sum()

ATTRIBUTION_METHODS = {
    'last_touch': attrib_last_touch,
    'first_touch': attrib_first_touch,
    'linear': attrib_linear,
    'time_decay': attrib_time_decay,
    'u_shape': attrib_u_shape,
    'w_shape': attrib_w_shape,
    'markov_proxy': attrib_markov_proxy,
    'shapley_proxy': attrib_shapley_proxy,
}

def run_method(join_df: pd.DataFrame, method_name: str) -> pd.DataFrame:
    f = ATTRIBUTION_METHODS[method_name]
    out = f(join_df).copy()
    out['method'] = method_name
    return out[['method', 'channel', 'credit']]

method_table_7d = pd.concat([run_method(joined_7d, m) for m in ATTRIBUTION_METHODS], ignore_index=True)
method_pivot_7d = method_table_7d.pivot_table(index='channel', columns='method', values='credit', aggfunc='sum').fillna(0)
display(method_pivot_7d)



method,first_touch,last_touch,linear,markov_proxy,shapley_proxy,time_decay,u_shape,w_shape
channel,,,,,,,,
73325,1.0,2.0,1.666667,1.567405,1.500000,1.766036,1.600000,1.600000
73327,1.0,1.0,1.000000,0.783703,1.000000,1.000000,1.000000,1.000000
73328,4.0,4.0,4.000000,3.134811,4.000000,4.000000,4.000000,4.000000
83677,1.0,1.0,1.000000,0.783703,1.000000,1.000000,1.000000,1.000000
408759,17.0,18.0,17.600000,14.106648,17.500000,17.850481,17.533333,17.250000
...,...,...,...,...,...,...,...,...
32323516,1.0,0.0,0.625000,1.567405,1.000000,0.394343,0.533333,0.520000
32368241,8.0,8.0,8.000000,6.269621,8.000000,8.000000,8.000000,8.000000
32368244,251.0,250.0,249.297034,203.762696,235.833333,250.149045,250.433531,247.879924


## 6) Robustness Checks: Windows and Lag Filters

We compare methods over:
- windows: 1, 3, 7, 14, 30 days
- minimum lag before conversion: 0h, 6h, 24h

> Side note: If channel credit changes a lot under small window/lag changes, the attribution policy may be fragile.


In [8]:
def build_join_table(exposures_df: pd.DataFrame, conversions_df: pd.DataFrame, lookback_days: int, min_lag_hours: float = 0.0) -> pd.DataFrame:
    merged = conversions_df.merge(exposures_df, on='user_id', how='left')
    delta_hours = (merged['ts_conversion'] - merged['ts_exposure']).dt.total_seconds() / 3600
    valid = merged[(delta_hours >= min_lag_hours) & (delta_hours <= lookback_days * 24)].copy()
    if valid.empty:
        return pd.DataFrame(columns=[
            'lookback_days','min_lag_hours','conversion_id','user_id','ts_conversion','conversion_value',
            'exposure_id','channel','ts_exposure','hours_before_conversion'
        ])
    valid['hours_before_conversion'] = delta_hours.loc[valid.index]
    valid['lookback_days'] = lookback_days
    valid['min_lag_hours'] = min_lag_hours
    cols=['lookback_days','min_lag_hours','conversion_id','user_id','ts_conversion','conversion_value','exposure_id','channel','ts_exposure','hours_before_conversion']
    return valid[cols].sort_values(['conversion_id','ts_exposure']).reset_index(drop=True)

windows = [1, 3, 7, 14, 30]
lags = [0, 6, 24]
robust_rows = []
for w in windows:
    for lag in lags:
        j = build_join_table(exposures, conversions, lookback_days=w, min_lag_hours=lag)
        if j.empty:
            continue
        conv_cov = j['conversion_id'].nunique()
        total_conv = conversions['conversion_id'].nunique()
        for m in ATTRIBUTION_METHODS:
            out = run_method(j, m)
            total_credit = float(out['credit'].sum())
            top_channel = out.sort_values('credit', ascending=False)['channel'].iloc[0] if len(out) else None
            robust_rows.append({
                'window_days': w,
                'min_lag_hours': lag,
                'method': m,
                'total_credit': total_credit,
                'coverage_pct': round(100 * conv_cov / max(total_conv, 1), 2),
                'top_channel': top_channel
            })

robustness = pd.DataFrame(robust_rows)
display(robustness.head(20))

stability = robustness.groupby('method').agg(
    avg_coverage_pct=('coverage_pct', 'mean'),
    top_channel_nunique=('top_channel', 'nunique'),
    total_credit_std=('total_credit', 'std')
).reset_index().sort_values(['top_channel_nunique', 'total_credit_std'])
display(stability)


,window_days,min_lag_hours,method,total_credit,coverage_pct,top_channel
0,1,0,last_touch,2507.0,65.97,32368244
1,1,0,first_touch,2507.0,65.97,32368244
2,1,0,linear,2507.0,65.97,32368244
3,1,0,time_decay,2507.0,65.97,32368244
4,1,0,u_shape,2507.0,65.97,32368244
5,1,0,w_shape,2480.4,65.97,32368244
6,1,0,markov_proxy,2507.0,65.97,32368244
7,1,0,shapley_proxy,2507.0,65.97,32368244
8,1,6,last_touch,998.0,26.26,10341182
9,1,6,first_touch,998.0,26.26,10341182


,method,avg_coverage_pct,top_channel_nunique,total_credit_std
7,w_shape,72.311429,2,756.872361
0,first_touch,72.311429,2,762.508800
1,last_touch,72.311429,2,762.508800
2,linear,72.311429,2,762.508800
4,shapley_proxy,72.311429,2,762.508800
6,u_shape,72.311429,2,762.508800
5,time_decay,72.311429,2,762.508800
3,markov_proxy,72.311429,2,762.508800


## 7) Segment-Level Sensitivity

Segments used:
- user touch-count cohort (low / medium / high)
- dominant channel cohort
- campaign cohort (bucketed campaign IDs)

> Side note: Segment sensitivity tells us whether conclusions hold across different user/channel contexts.


In [9]:
# Build user-level segment labels from real data
user_touch = exposures.groupby('user_id').size().rename('touch_count').reset_index()
# Robust cohorting without qcut edge-duplication failures
touch_pct = user_touch['touch_count'].rank(method='average', pct=True)
user_touch['touch_cohort'] = pd.cut(touch_pct, bins=[0.0, 1/3, 2/3, 1.0], labels=['low', 'medium', 'high'], include_lowest=True)

dom = exposures.groupby(['user_id', 'channel']).size().rename('n').reset_index()
dom = dom.sort_values(['user_id', 'n'], ascending=[True, False]).drop_duplicates('user_id')
dom = dom[['user_id', 'channel']].rename(columns={'channel': 'dominant_channel'})

camp_tmp = exposures.copy()
camp_tmp['campaign_cohort'] = camp_tmp['channel'].astype(str).str[-1]  # real-data-based lightweight cohort key
camp_mode = camp_tmp.groupby(['user_id', 'campaign_cohort']).size().rename('n').reset_index()
camp_mode = camp_mode.sort_values(['user_id', 'n'], ascending=[True, False]).drop_duplicates('user_id')
camp_mode = camp_mode[['user_id', 'campaign_cohort']]

user_segments = user_touch.merge(dom, on='user_id', how='left').merge(camp_mode, on='user_id', how='left')
display(user_segments.head(10))

joined_seg = joined_7d.merge(user_segments, on='user_id', how='left')

def run_segment_slice(df: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    rows = []
    for seg_val, g in df.groupby(segment_col, dropna=False):
        if g.empty:
            continue
        for m in ATTRIBUTION_METHODS:
            out = run_method(g, m)
            out['segment_col'] = segment_col
            out['segment_value'] = str(seg_val)
            rows.append(out)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['method', 'channel', 'credit', 'segment_col', 'segment_value'])

seg_results = pd.concat([
    run_segment_slice(joined_seg, 'touch_cohort'),
    run_segment_slice(joined_seg, 'dominant_channel'),
    run_segment_slice(joined_seg, 'campaign_cohort')
], ignore_index=True)

seg_summary = seg_results.groupby(['segment_col', 'segment_value', 'method'], as_index=False)['credit'].sum()
display(seg_summary.head(30))

# Top channel by segment/method
ranked = seg_results.sort_values(['segment_col', 'segment_value', 'method', 'credit'], ascending=[True, True, True, False])
top_by_segment = ranked.groupby(['segment_col', 'segment_value', 'method'], as_index=False).first()[['segment_col', 'segment_value', 'method', 'channel', 'credit']]
display(top_by_segment.head(30))



,user_id,touch_count,touch_cohort,dominant_channel,campaign_cohort
0,10011902,2,low,5544859,9
1,10015858,4,low,17686799,9
2,10036129,15,high,28351001,1
3,10070070,10,medium,13104554,4
4,10081786,3,low,10341182,2
5,10085074,2,low,9100693,3
6,10087416,9,medium,18975823,3
7,10087621,8,medium,27321366,6
8,10092925,28,high,27118781,1
9,10124895,1,low,10341182,2


/var/folders/r8/c7h0z2bn62x8452cd7vnf1v80000gn/T/ipykernel_26185/1656164484.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for seg_val, g in df.groupby(segment_col, dropna=False):


,segment_col,segment_value,method,credit
0,campaign_cohort,0,first_touch,285.0
1,campaign_cohort,0,last_touch,285.0
2,campaign_cohort,0,linear,285.0
3,campaign_cohort,0,markov_proxy,285.0
4,campaign_cohort,0,shapley_proxy,285.0
5,campaign_cohort,0,time_decay,285.0
6,campaign_cohort,0,u_shape,285.0
7,campaign_cohort,0,w_shape,281.6
8,campaign_cohort,1,first_touch,534.0
9,campaign_cohort,1,last_touch,534.0


,segment_col,segment_value,method,channel,credit
0,campaign_cohort,0,first_touch,15398570,66.000000
1,campaign_cohort,0,last_touch,15398570,67.000000
2,campaign_cohort,0,linear,15398570,65.709091
3,campaign_cohort,0,markov_proxy,15398570,53.876712
4,campaign_cohort,0,shapley_proxy,15398570,63.416667
5,campaign_cohort,0,time_decay,15398570,66.537429
6,campaign_cohort,0,u_shape,15398570,66.372222
7,campaign_cohort,0,w_shape,15398570,65.222549
8,campaign_cohort,1,first_touch,15184511,190.000000
9,campaign_cohort,1,last_touch,15184511,187.000000


## 8) Measurement-Ready Output Table

This table is the core exposure->conversion dataset that downstream dashboards/modeling can consume.

> Side note: In production this is often persisted in a warehouse/Delta table with versioned logic.


In [10]:
measurement_table = build_join_table(exposures, conversions, lookback_days=7, min_lag_hours=0)
print('Rows:', len(measurement_table))
print('Unique conversions:', measurement_table['conversion_id'].nunique())
display(measurement_table.head(15))


Rows: 17187
Unique conversions: 3395


,lookback_days,min_lag_hours,conversion_id,user_id,ts_conversion,conversion_value,exposure_id,channel,ts_exposure,hours_before_conversion
0,7,0,0,7306395,1970-01-17 18:33:13,1.0,12299,29427842,1970-01-10 22:32:07,164.018333
1,7,0,0,7306395,1970-01-17 18:33:13,1.0,13147,29427842,1970-01-12 10:14:25,128.313333
2,7,0,0,7306395,1970-01-17 18:33:13,1.0,14882,29427842,1970-01-14 10:14:41,80.308889
3,7,0,0,7306395,1970-01-17 18:33:13,1.0,17027,29427842,1970-01-16 23:31:42,19.025278
4,7,0,0,7306395,1970-01-17 18:33:13,1.0,17514,29427842,1970-01-17 16:21:13,2.200000
5,7,0,1,6013772,1970-01-02 14:27:49,1.0,1,15184511,1970-01-01 00:00:25,38.456667
6,7,0,1,6013772,1970-01-02 14:27:49,1.0,1890,15184511,1970-01-01 14:15:22,24.207500
7,7,0,1,6013772,1970-01-02 14:27:49,1.0,3697,15184511,1970-01-02 09:09:54,5.298611
8,7,0,1,6013772,1970-01-02 14:27:49,1.0,3702,15184511,1970-01-02 09:14:57,5.214444
9,7,0,1,6013772,1970-01-02 14:27:49,1.0,4158,15184511,1970-01-02 14:27:30,0.005278


## 8b) Important Note: Attribution Is Not Incrementality

> Side note for beginners: attribution assigns credit among observed touches, but it does not prove causal lift by itself.

Attribution answers: "which touch gets credit for this conversion path?"
Incrementality answers: "would conversion still happen without this touch?"

Use attribution for measurement and reporting workflow. Use uplift/causal methods for incrementality decisions.


## 9) Key Learning

- Real-world exposure->conversion joining is now explicit and auditable.
- Attribution method choice, window choice, and lag assumptions all materially affect channel credit.
- Segment-level sensitivity highlights where a method is stable or unstable across user/channel contexts.
- This notebook now covers baseline and next-level attribution depth in one reproducible workflow.

> Beginner reminder: use this notebook to validate measurement assumptions before using ROI dashboards for budget decisions.
